In [61]:
# Manipulación de datos
import numpy as np
import pandas as pd

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Estadística
import scipy.stats as stats
import statsmodels.api as sm

# Preprocesamiento y selección de modelos
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Modelos
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.svm import SVC, SVR

# Métricas
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
    mean_squared_error, mean_absolute_error, r2_score,
)

import warnings
warnings.filterwarnings("ignore")

%matplotlib inline
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

In [62]:
data = pd.read_csv("datos/df_clean.csv")
data.info()
data.head(20)

<class 'pandas.DataFrame'>
RangeIndex: 7500 entries, 0 to 7499
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   edad                       7500 non-null   int64  
 1   genero                     7500 non-null   str    
 2   horas_diarias_pantalla     7500 non-null   float64
 3   horas_redes_sociales       7500 non-null   float64
 4   horas_videojuegos          7500 non-null   float64
 5   horas_trabajo/estudio      7500 non-null   float64
 6   horas_sueño                7500 non-null   float64
 7   notificaciones_por_dia     7500 non-null   int64  
 8   aperturas_apps_por_dia     7500 non-null   int64  
 9   tiempo_pantalla_finde      7500 non-null   float64
 10  nivel_estres               7500 non-null   str    
 11  impacto_academico/laboral  7500 non-null   str    
 12  etiqueta_de_adiccion       7500 non-null   int64  
 13  horas_ocio                 7500 non-null   float64
 14  rat

,edad,genero,horas_diarias_pantalla,horas_redes_sociales,horas_videojuegos,horas_trabajo/estudio,horas_sueño,notificaciones_por_dia,aperturas_apps_por_dia,tiempo_pantalla_finde,nivel_estres,impacto_academico/laboral,etiqueta_de_adiccion,horas_ocio,ratio_finde_diario,estado_adiccion,diferencia_finde
0,21,Male,3.23,2.01,0.89,4.55,7.55,248,154,3.95,Medium,Yes,0,2.90,1.222910,No adicto,0.72
1,24,Other,5.09,3.81,2.24,4.44,7.66,127,71,6.71,Medium,Yes,0,6.05,1.318271,No adicto,1.62
2,31,Other,6.06,1.36,3.83,2.35,4.92,44,106,8.68,High,No,0,5.19,1.432343,No adicto,2.62
3,32,Other,7.83,5.85,1.51,3.54,8.23,178,107,9.77,High,Yes,1,7.36,1.247765,Adicto,1.94
4,25,Male,9.96,5.92,3.42,5.27,6.21,136,177,12.55,Low,No,1,9.34,1.260040,Adicto,2.59
5,26,Male,9.32,4.26,0.29,3.99,6.90,82,56,10.98,Medium,Yes,1,4.55,1.178112,Adicto,1.66
6,25,Male,10.40,4.93,1.60,0.86,8.61,165,95,11.43,Medium,No,1,6.53,1.099038,Adicto,1.03
7,26,Male,4.26,4.60,2.16,4.61,6.43,169,117,5.66,Low,No,1,6.76,1.328638,Adicto,1.40
8,21,Other,4.38,1.38,2.72,3.78,6.23,172,134,6.20,High,Yes,0,4.10,1.415525,No adicto,1.82
9,35,Other,9.76,4.73,1.36,2.11,5.21,20,82,12.69,Low,Yes,1,6.09,1.300205,Adicto,2.93


### Aplicando label encoding para las variables booleanas:

In [63]:
data['impacto_academico/laboral'] = data['impacto_academico/laboral'].map({'No': 0, 'Yes': 1})
data['estado_adiccion'] = data['estado_adiccion'].map({'No adicto': 0, 'Adicto': 1})
data.drop('etiqueta_de_adiccion', axis=1, inplace=True) #Eliminamos la columna etiqueta de adiccion que sería suplantada por 'estado_adiccion'


print("DataFrame después de Label Encoding:")
data.head()

DataFrame después de Label Encoding:


,edad,genero,horas_diarias_pantalla,horas_redes_sociales,horas_videojuegos,horas_trabajo/estudio,horas_sueño,notificaciones_por_dia,aperturas_apps_por_dia,tiempo_pantalla_finde,nivel_estres,impacto_academico/laboral,horas_ocio,ratio_finde_diario,estado_adiccion,diferencia_finde
0,21,Male,3.23,2.01,0.89,4.55,7.55,248,154,3.95,Medium,1,2.90,1.222910,0,0.72
1,24,Other,5.09,3.81,2.24,4.44,7.66,127,71,6.71,Medium,1,6.05,1.318271,0,1.62
2,31,Other,6.06,1.36,3.83,2.35,4.92,44,106,8.68,High,0,5.19,1.432343,0,2.62
3,32,Other,7.83,5.85,1.51,3.54,8.23,178,107,9.77,High,1,7.36,1.247765,1,1.94
4,25,Male,9.96,5.92,3.42,5.27,6.21,136,177,12.55,Low,0,9.34,1.260040,1,2.59


### Aplicamos One-Hot Encoding para `nivel_estres`:

In [64]:
data = pd.get_dummies(data, columns=['nivel_estres'], drop_first=True)
data['nivel_estres_Low'] = data['nivel_estres_Low'].astype(int)
data['nivel_estres_Medium'] = data['nivel_estres_Medium'].astype(int)

print("DataFrame después de One-Hot Encoding:")
data.head(20)

DataFrame después de One-Hot Encoding:


,edad,genero,horas_diarias_pantalla,horas_redes_sociales,horas_videojuegos,horas_trabajo/estudio,horas_sueño,notificaciones_por_dia,aperturas_apps_por_dia,tiempo_pantalla_finde,impacto_academico/laboral,horas_ocio,ratio_finde_diario,estado_adiccion,diferencia_finde,nivel_estres_Low,nivel_estres_Medium
0,21,Male,3.23,2.01,0.89,4.55,7.55,248,154,3.95,1,2.90,1.222910,0,0.72,0,1
1,24,Other,5.09,3.81,2.24,4.44,7.66,127,71,6.71,1,6.05,1.318271,0,1.62,0,1
2,31,Other,6.06,1.36,3.83,2.35,4.92,44,106,8.68,0,5.19,1.432343,0,2.62,0,0
3,32,Other,7.83,5.85,1.51,3.54,8.23,178,107,9.77,1,7.36,1.247765,1,1.94,0,0
4,25,Male,9.96,5.92,3.42,5.27,6.21,136,177,12.55,0,9.34,1.260040,1,2.59,1,0
5,26,Male,9.32,4.26,0.29,3.99,6.90,82,56,10.98,1,4.55,1.178112,1,1.66,0,1
6,25,Male,10.40,4.93,1.60,0.86,8.61,165,95,11.43,0,6.53,1.099038,1,1.03,0,1
7,26,Male,4.26,4.60,2.16,4.61,6.43,169,117,5.66,0,6.76,1.328638,1,1.40,1,0
8,21,Other,4.38,1.38,2.72,3.78,6.23,172,134,6.20,1,4.10,1.415525,0,1.82,0,0
9,35,Other,9.76,4.73,1.36,2.11,5.21,20,82,12.69,1,6.09,1.300205,1,2.93,1,0


Ahora procederemos a limpiar el dataset para quedarnos unicamente con las variables que sirvan para el modelo predictivo:

In [65]:
data.drop('tiempo_pantalla_finde', axis=1, inplace=True) #Aporta informacion muy similar a la pantalla diaria
data.drop('ratio_finde_diario', axis=1, inplace=True) # relacionado a tiempo_pantalla_finde
data.drop('diferencia_finde', axis=1, inplace=True) # relacionado a tiempo_pantalla_finde
data.drop('horas_ocio', axis=1, inplace=True) #Unicamente para explorar, tiene menos relacion que redes sociales sola
data.head()

,edad,genero,horas_diarias_pantalla,horas_redes_sociales,horas_videojuegos,horas_trabajo/estudio,horas_sueño,notificaciones_por_dia,aperturas_apps_por_dia,impacto_academico/laboral,estado_adiccion,nivel_estres_Low,nivel_estres_Medium
0,21,Male,3.23,2.01,0.89,4.55,7.55,248,154,1,0,0,1
1,24,Other,5.09,3.81,2.24,4.44,7.66,127,71,1,0,0,1
2,31,Other,6.06,1.36,3.83,2.35,4.92,44,106,0,0,0,0
3,32,Other,7.83,5.85,1.51,3.54,8.23,178,107,1,1,0,0
4,25,Male,9.96,5.92,3.42,5.27,6.21,136,177,0,1,1,0
